In [ ]:
from dataclasses import dataclass, field
from typing import Any

# --- SCHRITT 1: Mocks für die Backend-Klassen ---
# (Simuliert dataland_backend & dataland_qa, falls du lokal keine Verbindung hast)
# Wenn du im echten Environment bist, kannst du diesen Block überspringen und echt importieren!


@dataclass
class MockDataPoint:
    value: float | None = None
    data_source: Any = None


@dataclass
class MockScope1:
    scope1_ghg_emissions_in_tonnes: MockDataPoint = field(default_factory=MockDataPoint)


@dataclass
class MockGHG:
    scope1_ghg_emissions: Any | None = None  # Für den Report Output
    # Für Input Data Struktur:
    scope1_ghg_emissions_in_tonnes: MockDataPoint = field(default_factory=MockDataPoint)


@dataclass
class MockEnvironmental:
    greenhouse_gas_emissions: MockGHG = field(default_factory=MockGHG)
    water: Any = field(
        default_factory=lambda: type(
            "MockWater", (), {"emissions_to_water": None, "emissions_to_water_amount": MockDataPoint(10.0)}
        )()
    )
    waste: Any = field(default_factory=lambda: type("MockWaste", (), {"hazardous_waste_ratio": None})())


@dataclass
class MockSocial:
    social_and_employee_matters: Any = field(
        default_factory=lambda: type("MockSocialMatters", (), {"unadjusted_gender_pay_gap": None})()
    )


@dataclass
class SfdrData:
    environmental: MockEnvironmental = field(default_factory=MockEnvironmental)
    social: MockSocial = field(default_factory=MockSocial)


# --- SCHRITT 2: Deine echten Klassen laden ---
# (Hier importierst du normalerweise deine Dateien. Für den Test definieren wir sie inline oder laden sie)

# Monkeypatching für den Test, damit wir kein echtes OpenAI brauchen
from dataland_qa_lab.review.report_generator.sfdr_numeric_value_generator import SFDRNumericValueGenerator
from dataland_qa_lab.review.report_generator.sfdr_report_generator import SfdrReportGenerator
from dataland_qa_lab.utils.sfdr_data_collection import SFDRDataCollection


def mock_get_generic_value(text, kpi, unit):
    print(f"🤖 AI Mock gefragt nach: {kpi} ({unit})")
    if "Scope 1" in kpi:
        return 15000.0
    if "Water" in kpi:
        return 12.5
    if "waste" in kpi:
        return 5.0
    if "Pay Gap" in kpi:
        return 11.0
    return -1.0


SFDRNumericValueGenerator.get_generic_numeric_value = mock_get_generic_value

# --- SCHRITT 3: Testlauf ---

print("--- 1. Erstelle Mock-Daten (Simuliere Dataland API) ---")
# Wir simulieren einen Datensatz, der schon Werte in Dataland hat
input_data = SfdrData()
# Setze einen Dataland-Wert für Scope 1
input_data.environmental.greenhouse_gas_emissions.scope1_ghg_emissions_in_tonnes.value = 15000.0

collection = SFDRDataCollection(input_data)
print("Data Collection initialisiert.")
print(f"Dataland Scope 1 Wert: {collection.get_scope1_ghg_emissions()}")

print("\n--- 2. Starte Report Generator ---")
generator = SfdrReportGenerator()
relevant_pages_text = "Mock Text: Scope 1 is 15000 tCO2e. Water emissions 12.5 t. Hazardous waste 5%. Pay gap 11%."

# Hier läuft deine Logik ab!
report = generator.generatereport(relevant_pages_text, collection)

print("\n--- 3. Inspiziere das Ergebnis (Struktur-Check) ---")


def print_verdict(name, obj):
    if obj:
        print(
            f"✅ {name}: {obj.verdict} | Comment: {obj.comment} | Value: {getattr(obj.correctedData, 'value', 'N/A')}"
        )
    else:
        print(f"❌ {name}: Objekt ist None!")


# Prüfen wir die tiefen Pfade
print_verdict("Scope 1 GHG", report.environmental.greenhouse_gas_emissions.scope1_ghg_emissions)
print_verdict("Water Emissions", report.environmental.water.emissions_to_water)
print_verdict("Hazardous Waste", report.environmental.waste.hazardous_waste_ratio)
print_verdict("Gender Pay Gap", report.social.social_and_employee_matters.unadjusted_gender_pay_gap)

print("\n--- 4. Fertiges Objekt Dump ---")
# Da Pydantic Objekte eine .dict() oder .model_dump() Methode haben, simulieren wir das hier für den Output
try:
    # Versuche Pydantic v2 oder v1 Dump
    import pprint

    pprint.pprint(report.__dict__)
except:
    print("Konnte Objekt nicht dumpen, aber Tests oben waren erfolgreich.")

ImportError: cannot import name 'SfdrSocialEmployeeMatters' from 'dataland_qa.models' (/workspace/.venv/lib/python3.14/site-packages/dataland_qa/models/__init__.py)

In [3]:
import dataland_qa.models

print("🔍 Suche nach 'Social' Klassen in dataland_qa.models...")

# Hole alle Attribute aus dem Modul
all_attributes = dir(dataland_qa.models)

# Filter nach "Social"
social_classes = [name for name in all_attributes if "Social" in name]

if not social_classes:
    print("❌ Keine Klassen mit 'Social' gefunden. Ist das Paket richtig installiert?")
else:
    print("✅ Gefundene Klassen:")
    for name in sorted(social_classes):
        print(f"  - {name}")

print("\nBitte nutze den exakten Namen aus dieser Liste für den Import in sfdr_report_generator.py")

🔍 Suche nach 'Social' Klassen in dataland_qa.models...
✅ Gefundene Klassen:
  - SfdrSocial
  - SfdrSocialAntiCorruptionAndAntiBribery
  - SfdrSocialGreenSecurities
  - SfdrSocialHumanRights
  - SfdrSocialSocialAndEmployeeMatters

Bitte nutze den exakten Namen aus dieser Liste für den Import in sfdr_report_generator.py


In [4]:
import dataland_qa.models

print("🔍 Suche nach 'ExtendedDataPoint' Klassen in dataland_qa.models...")

# Alle Attribute holen
attributes = dir(dataland_qa.models)

# Filter nach "ExtendedDataPoint"
extended_classes = [a for a in attributes if "ExtendedDataPoint" in a]

if not extended_classes:
    print("❌ Keine ExtendedDataPoint Klassen gefunden.")
else:
    print(f"✅ Gefunden ({len(extended_classes)}):")
    # Wir suchen besonders nach Begriffen wie Decimal, Float, Number, Numeric
    priorities = ["Decimal", "Float", "Number", "Numeric", "Int"]

    for name in sorted(extended_classes):
        # Highlighte wahrscheinliche Kandidaten
        prefix = "👉 " if any(p in name for p in priorities) else "   "
        print(f"{prefix}{name}")

print("\nKopiere den passenden Klassennamen (z.B. ExtendedDataPointFloat) für den Fix!")

🔍 Suche nach 'ExtendedDataPoint' Klassen in dataland_qa.models...
✅ Gefunden (34):
👉 ExtendedDataPointBigDecimal
👉 ExtendedDataPointBigInteger
   ExtendedDataPointEutaxonomyFinancialsGeneralGeneralFiscalYearDeviationOptions
   ExtendedDataPointEutaxonomyNonFinancialsGeneralFiscalYearDeviationOptions
   ExtendedDataPointListEuTaxonomyActivity
   ExtendedDataPointListEuTaxonomyAlignedActivity
   ExtendedDataPointLocalDate
   ExtendedDataPointNuclearAndGasAlignedDenominator
   ExtendedDataPointNuclearAndGasAlignedNumerator
   ExtendedDataPointNuclearAndGasEligibleButNotAligned
   ExtendedDataPointNuclearAndGasNonEligible
   ExtendedDataPointPcafGeneralCompanyCompanyExchangeStatusOptions
   ExtendedDataPointPcafGeneralCompanyMainPcafSectorOptions
   ExtendedDataPointPcafGeneralGeneralFiscalYearDeviationOptions
   ExtendedDataPointSfdrGeneralGeneralFiscalYearDeviationOptions
   ExtendedDataPointYesNo
   ExtendedDataPointYesNoNa
👉 QaReportDataPointExtendedDataPointBigDecimal
👉 QaReportDataPo

In [5]:
from dataland_qa.models import SfdrEnvironmentalWater

print("🔍 Untersuche Felder von SfdrEnvironmentalWater...")

# Wir holen alle Felder (Pydantic v1 oder v2)
try:
    # Pydantic v1
    fields = SfdrEnvironmentalWater.__fields__.keys()
except AttributeError:
    # Pydantic v2
    fields = SfdrEnvironmentalWater.model_fields.keys()

print(f"✅ Verfügbare Felder ({len(fields)}):")
for name in sorted(fields):
    print(f"  - {name}")

print("\n👉 Kopiere den exakten Namen (wahrscheinlich 'emissions_to_water' oder ähnlich)!")

🔍 Untersuche Felder von SfdrEnvironmentalWater...
✅ Verfügbare Felder (6):
  - emissions_to_water_in_tonnes
  - high_water_stress_area_exposure
  - relative_water_usage_in_cubic_meters_per_million_eur_revenue
  - water_consumption_in_cubic_meters
  - water_management_policy
  - water_reused_in_cubic_meters

👉 Kopiere den exakten Namen (wahrscheinlich 'emissions_to_water' oder ähnlich)!


/tmp/ipykernel_26381/1902374142.py:9: PydanticDeprecatedSince20: The `__fields__` attribute is deprecated, use the `model_fields` class property instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  fields = SfdrEnvironmentalWater.__fields__.keys()


In [6]:
from dataland_qa.models import SfdrEnvironmentalWaste, SfdrSocialSocialAndEmployeeMatters


def print_fields(model_cls):
    print(f"\n🔍 Felder für {model_cls.__name__}:")
    try:
        # Pydantic v2
        fields = model_cls.model_fields.keys()
    except AttributeError:
        # Pydantic v1 Fallback
        fields = model_cls.__fields__.keys()

    for f in sorted(fields):
        print(f"  - {f}")


print("Untersuche verbleibende Modelle...")
print_fields(SfdrEnvironmentalWaste)
print_fields(SfdrSocialSocialAndEmployeeMatters)

Untersuche verbleibende Modelle...

🔍 Felder für SfdrEnvironmentalWaste:
  - hazardous_and_radioactive_waste_in_tonnes
  - non_recycled_waste_in_tonnes

🔍 Felder für SfdrSocialSocialAndEmployeeMatters:
  - average_gross_hourly_earnings_female_employees
  - average_gross_hourly_earnings_male_employees
  - board_gender_diversity_board_of_directors_in_percent
  - board_gender_diversity_supervisory_board_in_percent
  - controversial_weapons_exposure
  - corruption_legal_proceedings
  - environmental_policy
  - excessive_ceo_pay_ratio
  - fair_business_marketing_advertising_policy
  - fair_competition_policy
  - female_board_members_board_of_directors
  - female_board_members_supervisory_board
  - grievance_handling_mechanism
  - human_rights_due_diligence_policy
  - human_rights_legal_proceedings
  - ilo_core_labour_standards
  - iso14001_certificate
  - male_board_members_board_of_directors
  - male_board_members_supervisory_board
  - oecd_guidelines_for_multinational_enterprises_grievanc

In [1]:
from dataclasses import dataclass, field
from typing import Any

# Wir müssen die Backend-Daten simulieren, da wir keinen echten API-Zugriff im Notebook haben
# (Falls du lokal die echten Daten laden kannst, nutze dataset_provider.get_sfdr_dataset_by_id)


# 1. Mock für die Input-Daten (Das, was normalerweise aus der API kommt)
@dataclass
class MockDataSource:
    page: int = 7
    file_reference: str = "dummy_file.pdf"


@dataclass
class MockDataPoint:
    value: float | None = None
    data_source: MockDataSource = field(default_factory=MockDataSource)


# Baue die verschachtelte Struktur nach, die deine DataCollection erwartet
class MockSfdrData:
    def __init__(self):
        self.environmental = type("Env", (), {})()
        self.environmental.greenhouse_gas_emissions = type("GHG", (), {})()
        self.environmental.greenhouse_gas_emissions.scope1_ghg_emissions_in_tonnes = MockDataPoint(12345.0)

        self.environmental.water = type("Water", (), {})()
        self.environmental.water.emissions_to_water_in_tonnes = MockDataPoint(50.5)

        self.environmental.waste = type("Waste", (), {})()
        self.environmental.waste.hazardous_and_radioactive_waste_in_tonnes = MockDataPoint(10.0)

        self.social = type("Social", (), {})()
        self.social.social_and_employee_matters = type("Matters", (), {})()
        self.social.social_and_employee_matters.unadjusted_gender_pay_gap = MockDataPoint(12.0)


# 2. Deine echten Klassen laden
from dataland_qa_lab.review.report_generator.sfdr_numeric_value_generator import SFDRNumericValueGenerator
from dataland_qa_lab.review.report_generator.sfdr_report_generator import SfdrReportGenerator
from dataland_qa_lab.utils.sfdr_data_collection import SFDRDataCollection


# 3. KI Mocken (Damit wir keine echten Credits verbrauchen und das Ergebnis steuern)
# Wir tun so, als hätte die KI genau dieselben Werte im Text gefunden
def mock_get_generic_value(text, kpi, unit):
    if "Scope 1" in kpi:
        return 12345.0  # Match -> QAACCEPTED
    if "Water" in kpi:
        return 9999.0  # Mismatch -> QAREJECTED (zum Testen)
    return -1.0


# Den echten Call überschreiben (Monkey Patching)
SFDRNumericValueGenerator.get_generic_numeric_value = mock_get_generic_value

# 4. Ausführen
print("🚀 Starte Test-Generierung...")
mock_dataset = MockSfdrData()
collection = SFDRDataCollection(mock_dataset)
generator = SfdrReportGenerator()

# Wir simulieren Text
fake_text = "Scope 1 emissions: 12345.0 tonnes. Water: 9999.0 tonnes."
report = generator.generate_report(fake_text, collection)

# 5. Ergebnis inspizieren
print("\n📊 REPORT ERGEBNIS:")


# Helper zum Anzeigen
def show_field(category, field_name, obj):
    if not obj:
        print(f"❌ {category} -> {field_name}: LEER (None)")
        return

    status = "✅ MATCH" if obj.verdict == "QAACCEPTED" else "⚠️ ABWEICHUNG"
    ai_val = obj.correctedData.value if obj.correctedData else "N/A"
    print(f"{status} | {category:<15} | {field_name:<25} | Verdict: {obj.verdict} | AI Wert: {ai_val}")


print("-" * 100)
# Scope 1
show_field("Environmental", "Scope 1 GHG", report.environmental.greenhouse_gas_emissions.scope1_ghg_emissions_in_tonnes)
# Water
show_field("Environmental", "Water Emissions", report.environmental.water.emissions_to_water_in_tonnes)
# Waste
show_field(
    "Environmental", "Hazardous Waste", report.environmental.waste.hazardous_waste_ratio
)  # Achtung: Im Report Objekt heißt das Feld evtl noch anders im QA Model?
# Social
show_field("Social", "Gender Pay Gap", report.social.social_and_employee_matters.unadjusted_gender_pay_gap)
print("-" * 100)

print("\n💡 Wenn du hier grüne Haken siehst, funktioniert dein Code perfekt!")

🚀 Starte Test-Generierung...


TypeError: mock_get_generic_value() got an unexpected keyword argument 'ai_model'

In [ ]:
import sys
import os
from unittest.mock import patch, MagicMock

# 1. Pfad setup (damit Python deine Module findet)
# Wir fügen das 'src' Verzeichnis zum Python-Pfad hinzu
current_dir = os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
if src_path not in sys.path:
    sys.path.append(src_path)

print(f"Suche Module in: {src_path}")

# 2. Importiere deine Klassen
try:
    from dataland_qa_lab.review.sfdr_numeric_value_generator import SFDRNumericValueGenerator
    from dataland_qa_lab.prompting_services.sfdr_prompting_service import SFDRPromptingService
    print("✅ Module erfolgreich importiert.")
except ImportError as e:
    print(f"❌ Import Fehler: {e}")
    print("Bitte stelle sicher, dass du das Notebook im 'notebooks' Ordner ausführst oder den Pfad oben anpasst.")

# 3. TEST: Was kommt wirklich bei der KI an?
print("\n--- PROMPT DEBUGGING START ---")

# Unser Test-Markdown (Eindeutiger String)
TEST_MARKDOWN = """
### EXTRAHIERTER TEXT AUS PDF ###
Seite 7:
Die Scope 1 Emissionen betrugen im Jahr 2023 exakt 12.345 Tonnen CO2-Äquivalente.
Das Unternehmen hat keine Scope 2 Emissionen berichtet.
### ENDE TEXT ###
"""

# Wir 'patchen' (mocken) den Request, damit nichts an Azure gesendet wird
# Wir wollen nur sehen, womit die Funktion aufgerufen wurde.
with patch("dataland_qa_lab.review.generate_gpt_request.GenerateGptRequest.generate_gpt_request") as mock_gpt:
    
    # Dummy Rückgabe, damit der Code nicht abstürzt
    mock_gpt.return_value = ["12345.0"]

    # Führe die Extraktion aus
    print("Rufe get_scope1_emissions auf...")
    result = SFDRNumericValueGenerator.get_scope1_emissions(TEST_MARKDOWN)

    # Hole die Argumente, mit denen generate_gpt_request aufgerufen wurde
    # args[0] ist der Prompt, args[1] ist das Schema
    called_args, _ = mock_gpt.call_args
    generated_prompt = called_args[0]

    print("\n--- ERGEBNIS ANALYSE ---")
    
    # CHECK 1: Ist der Markdown im Prompt?
    if TEST_MARKDOWN in generated_prompt:
        print("✅ ERFOLG: Der Markdown-Text wurde im Prompt gefunden!")
    else:
        print("❌ FEHLER: Der Markdown-Text fehlt im Prompt!")

    # CHECK 2: Visuelle Prüfung
    print("\n--- DER GENERIERTE PROMPT (Ausschnitt) ---")
    print(generated_prompt)
    print("------------------------------------------")

    # Optional: Speichern in Datei zur vollen Ansicht
    with open("full_debug_prompt.txt", "w", encoding="utf-8") as f:
        f.write(generated_prompt)
    print("\nℹ️ Der vollständige Prompt wurde in 'full_debug_prompt.txt' gespeichert.")